# AWS Bedrock AgentCore Gateway 대상의 Bearer Token 주입 패턴

## 개요

고객은 MCP Client(Cyphr, Amazon Q)가 AgentCore Gateway를 통해 백엔드 대상(Lambda 함수 또는 RESTful 서비스)이 제공하는 도구(예: Asana 함수)를 호출할 수 있기를 원합니다. 이때 중요한 요구 사항은 다운스트림 서비스 인증을 위해 MCP Client의 동적 bearer token을 대상에 주입하는 것입니다. 핵심 과제는 AgentCore Gateway의 필수 아웃바운드 인증 요구 사항이 동적 헤더 주입과 충돌할 수 있다는 점입니다.

다음 아키텍처 다이어그램은 AgentCore 아웃바운드 인증 메커니즘과 충돌하지 않으면서 인증 토큰(예: "Bearer actual-auth-token")을 주입하기 위해 제안하는 솔루션을 보여 줍니다.

<img src="./images/token-injection-architecture.png" alt="토큰 주입 패턴을 보여 주는 아키텍처 다이어그램" title="Architecture diagram showing token injection pattern" />

이 솔루션에서 MCP Client는 도구가 사용할 인증 토큰을 가져와 페이로드의 파라미터로 AgentCore Gateway에 전달하고, 이 토큰은 다시 사용자 지정 헤더로 도구에 전달됩니다.

### 사전 요구 사항 배포

이 솔루션은 다음 구성 요소를 배포합니다. 
- 인바운드 인증을 위한 Cognito user pool 
- Lambda 통합이 적용된 API Gateway
- AgentCore 아웃바운드 인증을 위한 API Key


In [ ]:
cd prerequisites/agentcore-components

In [ ]:
!bash prereq.sh

## API Gateway를 대상으로 사용하는 AgentCore Gateway 생성




### 1단계: 종속성 설치 및 가져오기 

시작하기 전에 이 실습의 사전 요구 사항을 설치합니다.

In [ ]:
import os
import sys

# 현재 스크립트의 디렉터리 가져오기
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우의 대체 경로(예: Jupyter)

print(f"current_dir set to {current_dir}")

# utils.py가 있는 디렉터리로 이동("04-bearer-token-injection"까지 두 단계 위)
utils_dir = os.path.abspath(os.path.join(current_dir, "../.."))

# sys.path에 추가
sys.path.insert(0, utils_dir)

import utils

In [ ]:
# 필수 패키지 설치
%pip install -U -r ../../requirements.txt -q

### 2단계: AgentCore Gateway 생성

다음 Python 코드는 아래 작업을 수행합니다.
- AgentCore Gateway를 생성합니다.
- 앞에서 생성한 APIGateway를 API_KEY와 함께 대상으로 추가합니다.

In [ ]:
# agentcore_gateway_creation 모듈에서 함수 가져오기
import sys
import os

# 모듈을 가져올 수 있도록 현재 디렉터리를 Python 경로에 추가
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from agentcore_gateway_creation import create_agentcore_gateway

# 1단계: AgentCore Gateway 생성
print("🚀 Creating AgentCore Gateway...")
gateway = create_agentcore_gateway()
print(f"✅ Gateway created with ID: {gateway['id']}")
print(f"Gateway URL: {gateway['gateway_url']}")

### 3단계: APIGateway를 AgentCore Gateway의 대상으로 추가

먼저 AgentCore Gateway가 "READY" 상태인지 확인합니다.

In [ ]:
import time
import boto3

# Gateway가 ACTIVE 상태가 될 때까지 대기
gateway_client = boto3.client("bedrock-agentcore-control")
while True:
    response = gateway_client.get_gateway(gatewayIdentifier=gateway["id"])
    status = response["status"]
    print(f"Gateway status: {status}")
    if status == "READY":
        break
    elif status == "FAILED":
        raise Exception("Gateway creation failed")
    time.sleep(10)  # 다시 확인하기 전에 10초 대기

이제 Gateway 대상을 추가하여 설정을 완료합니다.

In [ ]:
from agentcore_gateway_creation import add_gateway_target

# 2단계: Gateway 대상 추가
print("🎯 Adding Gateway Target...")
add_gateway_target(gateway["id"])
print("✅ Gateway target configuration completed!")

이 대상을 정의하는 데 사용된 OpenAPI Spec을 살펴봅니다.

<img src="./images/bearer-token-in-api-spec.png" alt="API Spec에서 헤더로 정의된 bearer token" title="bearer token in api spec as a header" />

토큰은 스키마에서 헤더 파라미터로 정의되며 HTTP 헤더 ```X-Asana-Token```으로 전달됩니다. 이는 예시일 뿐이며, 통합 환경에 맞게 이와 같은 헤더를 정의할 수 있습니다.

HTTP 헤더는 인증 토큰에 가장 일반적으로 사용되는 패턴입니다.

### 4단계: 엔드 투 엔드 흐름 테스트

이제 MCP Client에서 ``` tools/list ```를 호출하는 엔드 투 엔드 흐름을 테스트합니다.

In [ ]:
import os
import boto3
import time
import uuid
import json
import requests

# 1단계: MCP Client가 AgentCore로 인증하는 데 사용할 인바운드 인증 토큰 가져오기
token_url = utils.get_ssm_parameter("/app/asana/demo/agentcoregwy/cognito_token_url")
client_id = utils.get_ssm_parameter("/app/asana/demo/agentcoregwy/machine_client_id")
client_secret = utils.get_cognito_client_secret()
inbound_auth_token = utils.fetch_access_token(client_id, client_secret, token_url)
print("access token received")

# 2단계: Gateway 구성을 가져오고 페이로드 준비
GATEWAY_MCP_URL = gateway_url = (
    f"https://{utils.get_ssm_parameter('/app/asana/demo/agentcoregwy/gateway_id')}.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
)

SESSION_ID = str(uuid.uuid4())
headers = {
    "Authorization": f"Bearer {inbound_auth_token}",  # 인바운드 인증용
    "Content-Type": "application/json",
    "Mcp-Session-Id": SESSION_ID,
}

# 3단계: 사용 가능한 도구를 가져오기 위해 tools/list 호출
list_body = {"jsonrpc": "2.0", "id": "list-1", "method": "tools/list"}

list_response = requests.post(GATEWAY_MCP_URL, headers=headers, json=list_body)
print(f"tools/list Status: {list_response.status_code}")
print("Available Tools:")
print(json.dumps(list_response.json(), indent=2))

### 5단계: MCP Client에서 AgentCore Gateway 대상으로 Bearer token 주입

이제 파라미터로 bearer token ``` Bearer ASANA-TOKEN ```을 전달하고(``` ASANA-TOKEN ```은 MCP Client가 실제로 가져온 토큰으로 대체), ```AgentCoreGwyAPIGatewayTarget___asanaInvoke``` 도구를 위의 ```tools/list``` 응답에서 호출합니다. 

In [ ]:
# 1단계: MCP Client가 AgentCore로 인증하는 데 사용할 인바운드 인증 토큰 가져오기
token_url = utils.get_ssm_parameter("/app/asana/demo/agentcoregwy/cognito_token_url")
client_id = utils.get_ssm_parameter("/app/asana/demo/agentcoregwy/machine_client_id")
client_secret = utils.get_cognito_client_secret()
inbound_auth_token = utils.fetch_access_token(client_id, client_secret, token_url)
print("access token received")

# 2단계: Gateway 구성을 가져오고 페이로드 준비
GATEWAY_MCP_URL = gateway_url = (
    f"https://{utils.get_ssm_parameter('/app/asana/demo/agentcoregwy/gateway_id')}.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
)

SESSION_ID = str(uuid.uuid4())
headers = {
    "Authorization": f"Bearer {inbound_auth_token}",  # 인바운드 인증용
    "Content-Type": "application/json",
    "Mcp-Session-Id": SESSION_ID,
}

# 3단계: 인수에 Asana 토큰을 포함하여 tools/call 호출(업데이트된 스키마에 따라 헤더로 전달)
call_body = {
    "jsonrpc": "2.0",
    "id": "call-1",
    "method": "tools/call",
    "params": {
        "name": "AgentCoreGwyAPIGatewayTarget___asanaInvoke",  # tools/list에 표시된 정확한 이름
        "arguments": {
            "tool_name": "createTask",
            "X-Asana-Token": "Bearer ASANA-TOKEN",  # 여기에 실제 Asana bearer token 입력(헤더로 전달됨)
            "name": "Test Task from MCP",
            "notes": "This is a test description",
            "project": "your-project-gid",  # 실제 Project GID로 대체
        },
    },
}

call_response = requests.post(GATEWAY_MCP_URL, headers=headers, json=call_body)
print(f"tools/call Status: {call_response.status_code}")
print("Tool Call Response:")
print(json.dumps(call_response.json(), indent=2))

### 6단계: 토큰 주입 확인

CloudWatch의 로그를 확인하여 토큰 주입을 검증합니다. 사전 요구 사항에서 생성한 ```AsanaIntegrationStackInfra``` Cloudformation 스택으로 이동합니다. 

```CloudFormation -> Stacks -> AsanaIntegrationStackInfra```에서 아래와 같이 Lambda 함수를 찾습니다.
<img src="./images/AgentCoreGwyAsanaIntegrationDemo_function_resource.png" alt="AgentCoreGwyAsanaIntegrationDemo 함수 리소스 " title="AgentCoreGwyAsanaIntegrationDemo function resource"/> 


Lambda 함수를 별도 창에서 열고 아래 스크린샷과 같이 ```View CloudWatch Logs```를 클릭합니다. <img src="./images/AgentCoreGwyAsanaIntegrationDemo_function_log_group.png" alt="AgentCoreGwyAsanaIntegrationDemo 함수 로그 그룹 " title="AgentCoreGwyAsanaIntegrationDemo function log group"/>  


다음 스크린샷은 AgentCore Gateway를 통한 ```AgentCoreGwyAPIGatewayTarget___asanaInvoke``` 도구 호출 로그를 보여 줍니다. OpenAPI Spec에 정의된 대로 ``` Bearer token ```이 MCP Client에서 AgentCore Gateway를 거쳐 Lambda 함수까지 ``` X-Asana-Token ``` 헤더로 전달된 것을 확인할 수 있습니다. 

<img src="./images/Lambda-log-received-headers.png" alt="헤더로 주입된 Bearer token" title="Bearer token injected as header" />


이를 통해 MCP Client가 가져온 인증 토큰을 Lambda 함수(또는 다른 도구)에서 사용하여 AgentCore 아웃바운드 인증과 충돌하지 않으면서 Asana와 같은 서드 파티 SaaS 통합으로 인증할 수 있음을 알 수 있습니다.


MCP Client에서 전달한 페이로드의 나머지 부분은 아래와 같이 이벤트 본문을 통해 Lambda 함수에서 사용할 수 있습니다.
<img src="./images/Lambda-log-received-body.png" alt="페이로드 본문" title="payload body" />

## 정리

### AgentCore Gateway 삭제

In [ ]:
import utils
import boto3

gateway_client = boto3.client("bedrock-agentcore-control")

gateway_id = "agentcore-gw-asana-integration"
utils.delete_gateway(gateway_client, gateway_id)

Deleting all targets for gateway agentcore-gw-asana-integration-gyhxiv6rt5


### Cloudformation 스택 삭제

In [ ]:
cd 02-AgentCore-gateway/04-bearer-token-injection

In [ ]:
!bash clean_up.sh